In [232]:
from copy import deepcopy
from itertools import groupby 
import numpy as np 
import matplotlib.pyplot as plt
import neuralnet as nn
from neuralnet.preprocess import prepare_data, hot_encode
from neuralnet.activations import step, derivative_htanh , relu, derivative_relu
from neuralnet.losses import l2_hinge_loss_gradient


j = complex(0,1)
π = np.pi
spin_half_values = [1/2,-1/2]

up = np.array([1.0,0],dtype=complex)
down = np.array([0.0,1.0],dtype=complex)


σx = np.array([[0,1],
               [1,0]], dtype=complex)

σy = np.array([[0,-1j],
               [1j,0]],dtype=complex)

σz = np.array([[1,0],
               [0,1]],dtype=complex)


def quantum_tanh(z,a, ϵ=1):

   """
   A "quantum" activation function, where z and a 

   """

   return np.tanh(z/a)

def measure(state_vectors: np.ndarray, eigenvalues: np.ndarray):
    """
    Perform a quantum measurement on a set of state vectors.

    Parameters:
    - state_vectors (np.ndarray): A 2D NumPy array of shape (N, M), where N is the number of state vectors
      and M is the number of possible eigenstates. Each row represents a quantum state in an eigenbasis.
    - eigenvalues (np.ndarray): A 1D NumPy array of shape (M,) containing the eigenvalues corresponding 
      to the measurement observable.

    Returns:
    - measurements (np.ndarray): A 1D NumPy array of shape (N,) containing the measured eigenvalues 
      for each state vector.
    - projected_states (np.ndarray): A 2D NumPy array of shape (N, M) representing the post-measurement 
      states (collapsed wavefunctions) after measurement.
    """

    eig_index = np.arange(len(eigenvalues))  # Indices of eigenvalues
    # Measurements: stores the results (e.g., energy, spin) for each state
    measurements = np.zeros(len(state_vectors), dtype=float)
    # Projected states: stores the new wavefunctions after measurement
    projected_states = np.zeros((len(state_vectors), len(eigenvalues)), dtype=complex)
    for i in range(len(state_vectors)): 
        # Compute probability distribution from the state vector amplitudes
        ps = np.abs(state_vectors[i])**2  # Probabilities must be real and non-negative
        # Randomly choose an eigenvalue index according to probability distribution
        index = np.random.choice(eig_index, p=ps) 
        # Store the measured eigenvalue
        measurements[i] = eigenvalues[index]
        # Collapse the wavefunction: Set to the measured eigenstate
        projected_states[i] = np.zeros(len(eigenvalues), dtype=complex)  # Reset state
        projected_states[i, index] = 1.0 + 0.0j  # Collapse to the measured eigenstate

    return measurements, projected_states

def rotate(measurements: np.ndarray[:],
           states: np.ndarray[:],
           W: list[np.ndarray[:,:] ], 
           σ: callable = quantum_tanh ,
           a: (int | float ) = .5) -> np.ndarray[:,:]: 
   """
    Applies a quantum-inspired rotation to a set of quantum states based on given measurements.

    Parameters:
    - measurements (np.ndarray): A 1D NumPy array containing the measured values of the quantum system.
    - states (np.ndarray): A 2D NumPy array of shape (N, M) representing the quantum states, 
      where N is the number of states and M is the number of spin components.
    - W (list of np.ndarray): A list of 2D NumPy arrays representing weight matrices that determine
      the interaction between measurements and state rotations.
    - σ (callable, optional): A function that applies a nonlinear transformation to the sum of weighted measurements.
      Defaults to `quantum_tanh`.
    - a (int | float, optional): A scaling parameter for the nonlinear function `σ`. Default is 0.5.

    Returns:
    - new_states (np.ndarray): A 2D NumPy array of shape (N, M), representing the rotated quantum states.

    Functionality:
    1. Computes the **rotation angles** \( \theta \) based on the measurements and a nonlinear transformation.
    2. Constructs **rotation matrices** using the computed angles.
    3. Applies the rotation to each quantum state.

    Notes:
    - Uses the **Pauli-Y matrix** \( \sigma_y \) for spin rotations.
    - The function assumes **π is defined globally**.
    - The variable `j` is assumed to be the imaginary unit \( i \) (i.e., `1j` in Python).
  """
   # determine angle of rotation 
   new_states = np.zeros((W.shape[0],len(states[0])), dtype=complex)
   θ = π/2*(measurements-σ(np.sum(W@measurements),a))
   # actually rotate states 
   R = np.array([np.cos(θi/2)*np.eye(2)-j*np.sin(θi/2)*σy for θi in θ])
   for i in range(len(new_states)):
      new_states[i] = R[i]@states[i]

   return new_states

#TODO : "up" should not be a global variable
def intialize_up(W:list[np.ndarray[:,:]],state_list:list[np.ndarray]):

    """
    Initializes the state_list by setting all elements in each layer to 'up'.

    Parameters:
    -----------
    W : list[np.ndarray]
        A list of 2D numpy arrays, where each array represents a layer.
    state_list : list[np.ndarray]
        A list of 2D numpy arrays representing the states at each layer. The function modifies
        this list by updating elements at each layer.

    Returns:
    --------
    list[np.ndarray]
        The updated state_list with elements in layers (starting from index 1) set to 'up'.
    
    Notes:
    ------
    - Assumes `up` is a predefined variable.
    - Updates `state_list[l+1]`, meaning `state_list` should have `len(W) + 1` layers.
    - Ensure `state_list` has compatible dimensions before calling this function.
    """
    for l in range(len(W)): 
      for s in range(W[l].shape[0]):
          state_list[l][s, :] = up  
    return state_list 

def quantum_feedforward(x: np.ndarray[:,:],
                W: list[np.ndarray[:,:]], 
                σ: callable, 
                eigenvalues: (list | np.ndarray)):
  
  """
    Performs a quantum-inspired feedforward computation using measurement-based 
    quantum computation principles. The function propagates qubits, represented as 
    vectores, through a series of quantum-inspired transformations involving measurement, 
    rotation, and feedforward updates. The initial qubits are put through a linear classical step,
    the final qubits which determine output are also put through such a linear classical step. 

    Parameters:
    ----------
    x : np.ndarray[:, :]
        The input classical data (matrix of shape `(d, n)`, where `d` is the 
        input dimension and `n` is the number of samples).
    
    W : list[np.ndarray[:, :]]
        A list of weight matrices for each layer. Each `W[l]` is a matrix 
        determining how quantum states interact at layer `l`.
    
    σ : callable
        A nonlinear activation function applied at appropriate stages.
    
    eigenvalues : list | np.ndarray
        The eigenvalues corresponding to the measurement process, used 
        to extract classical information from quantum states.

    Returns:
    -------
    classical_output : np.ndarray[:, :]
        The final classical output after the quantum-inspired feedforward 
        processing, obtained by transforming the final quantum state 
        with the last weight matrix `W[-1]`.
    
  """
  # introduce classical step : 
  classical_bits = W[0] @ x
  classical_bits = classical_bits / np.linalg.norm(classical_bits, axis=1, keepdims=True)
  measurements, initial_states = measure(classical_bits,eigenvalues)

  # intialize all to up state at first 
  state_list = [initial_states] + [ np.zeros((W[l].shape[0],len(spin_half_values)),dtype=complex) for l in range(len(W))]
  state_list = intialize_up(W,state_list)

  # perform measure-rotate feedforward
  for l in range(1,len(W)-1): 
      states = rotate(measurements,initial_states,W[l])
      measurements, states = measure(states,spin_half_values)
      state_list[l] = states

    # final linear classical step
  classical_output = W[-1] @ state_list[-1]
  return classical_output

class QuantumNeuralNetwork(nn.NeuralNetwork):
    """
    A class for constructing and training a fully connected quantum neural network.

    Attributes:
        hidden_layer_n (int): Number of hidden layers in the network.
        layer_n (int): Total number of layers (input + hidden + output).
        layer_sizes (np.array): List of sizes for each layer in the network.
        bias (list): List of bias vectors for each layer.
        weights (list): List of weight matrices connecting the layers.
        activation_f (callable): Activation function (default: sigmoid).
        activation_df (callable): Derivative of the activation function.
        cost_function (callable): Cost function for training (if provided).

    Methods:
        train(minibatch=True, minibatch_pool=10, iterations=100, η=1e-6) -> 'NeuralNetwork':
            Trains the neural network using gradient descent.

    Parameters:
        hidden_layer (int): Number of hidden layers in the network.
        layer_sizes (list[int | float]): List of hidden layer sizes (default: [10]).
        activation_function (callable): Activation function for all layers (default: sigmoid).
        activation_derivative (callable): Derivative of the activation function (default: derivative_sigmoid).
        cost (callable): Cost function to minimize during training (optional).
        cost_grad (callable): Cost function gradient with respect to activations solely

    Train Method Parameters:
        minibatch (bool): Whether to use mini-batch gradient descent (default: True).
        minibatch_pool (int | float): Number of samples per mini-batch (default: 10).
        iterations (int | float): Number of training iterations (default: 100).
        η (int | float): Learning rate for gradient descent (default: 1e-6).

    Returns:
        NeuralNetwork: The trained neural network object.

    Example:
        nn = NeuralNetwork(
                           layer_sizes=[10,64, 32,1],
                           activation_function=sigmoid,
                           activation_derivative=derivative_sigmoid)
        nn.train(minibatch=True, minibatch_pool=32, iterations=1000, η=0.01)
    """

    def __init__(
        self,
        layer_sizes: (list[int] | list[float]),
        activation_function: callable,
        activation_derivative: callable,
        cost_function: callable,
        cost_grad: callable,
        eigenvalues: np.ndarray[:], 
        quantumness: (float | int)
    ) -> 'QuantumNeuralNetwork':
        
        super().__init__(layer_sizes,
                          activation_function,
                          activation_derivative,
                          cost_function,
                          cost_grad)

        self.quantumness = quantumness
        self.eigs = eigenvalues
        self.state_matrix = [ np.random.choice([]) ]

    def __str__(self):
        """
        Returns a string representation of the neural network's architecture, weights, and biases.
        """
        display_str = "Quantum Neural Network Structure:\n"
        display_str += f"Number of Layers: {self.layer_n}\n"
        display_str += f"Hidden Layers: {self.hidden_layer_n}\n"
        display_str += "Layer Sizes: " + " -> ".join(map(str, self.layer_sizes)) + "\n\n"

        display_str += "Biases:\n"
        for i, bias in enumerate(self.bias, start=1):
            display_str += f"  Layer {i + 1}: Shape {bias.shape}\n"

        display_str += "\nWeights:\n"
        for i, weight in enumerate(self.weights, start=1):
            display_str += f"  Layer {i}: Shape {weight.shape}\n"

        display_str += f"\nActivation Function: {self.activation_f.__name__}\n"
        display_str += f"Activation Derivative: {self.activation_df.__name__}\n"
        display_str += f"Cost Function: {self.cost.__name__}\n"
        display_str += f"Cost Gradient: {self.cost_grad.__name__}\n"

        return display_str
    
    def train(
        self,
        input,
        output,
        momentum: (int | float) = None,
        minibatch: bool = True,
        minibatch_pool: (int | float) = 10,
        iterations: (int | float) = 100,
        η: (int | float) = 1e-6,
    ) -> None:
        """
        Trains the neural network using gradient descent.

        Parameters:
            minibatch (bool): Whether to use mini-batch gradient descent (default: True).
            minibatch_pool (int | float): Size of the mini-batch for training (default: 10).
            iterations (int | float): Number of training iterations (default: 100).
            η (int | float): Learning rate for gradient descent (default: 1e-6).

        Returns:
            NeuralNetwork: The trained neural network object.

        Description:
            - Implements forward propagation for each input to compute activations.
            - Performs backpropagation to compute gradients for weights and biases.
            - Updates weights and biases using gradient descent.
            - Supports mini-batch gradient descent if `minibatch` is set to True.

        Example:
            nn.train(minibatch=True, minibatch_pool=32, iterations=1000, η=0.01)
        """

        for _ in range(iterations):
            if minibatch:
                indexes = np.random.choice(input.shape[0], size=minibatch_pool)
                X, Y = input[indexes], output[indexes]
            else:
                X, Y = input, output

            w_grads = [np.zeros(matrix.shape) for matrix in self.weights]

            b_grads = [np.zeros(vector.shape) for vector in self.bias]

            if momentum is not None:
                v = [np.zeros(w.shape) for w in self.weights]

            # iterate for each set of x and y
            # find zs and as (pre-act and activation)
            for x, y in zip(X, Y):
                # print("x id:",id(x))

                # def "quantum" feedforward
                z0 = self.weights[0] @ x + self.bias[0]
                zs = [z0]
                a0 = self.activation_f(z0)
                activations = [a0]
                for l in range(1, self.layer_n - 1, 1):
                    # print("layers:",l,l-1)
                    zl = self.weights[l] @ activations[l - 1] + self.bias[l]
                    activation = self.activation_f(zl)
                    # print("activation:", activation)
                    zs.append(zl)
                    activations.append(activation)

                z_output = zs[-1]
                a_output = activations[-1]
                output_error = self.cost_grad(a_output, y) * self.activation_df(
                    z_output
                )
                errors = [output_error]
                for l in range(self.hidden_layer_n, 0, -1):
                    error = (
                        self.weights[l].T @ errors[-1] * self.activation_df(zs[l - 1])
                    )
                    errors.append(error)

                errors.reverse()
                # compute sum of error
                for l in range(0, self.hidden_layer_n + 1, 1):
                    # w_grads[l] += errors[l]@activations[l].T
                    w_grads[l] += np.outer(
                        errors[l], activations[l - 1] if l > 0 else x
                    )
                    # print(w_grads)
                    b_grads[l] += errors[l]
                    # print(b_grads)

            # gradient descent
            if momentum is None:
                for l in range(0, self.hidden_layer_n + 1):
                    self.weights[l] -= η / minibatch_pool * w_grads[l]
                    self.bias[l] -= η / minibatch_pool * b_grads[l]
            else:
                γ = momentum
                for l in range(0, self.hidden_layer_n + 1):
                    v[l] = γ * v[l] + η / minibatch_pool * w_grads[l]
                    self.weights[l] -= v[l]
                    self.bias[l] -= η / minibatch_pool * b_grads[l]

    def predict(self, input: list[np.ndarray]) -> list[np.ndarray]:
        """
        Predicts the output for a given input using the trained neural network.

        Parameters:
            input (np.array): Input data to predict, where each row corresponds to a single input instance.

        Returns:
            list: A list of predictions where each prediction corresponds to the output of the neural network
                  for the corresponding input instance.

        Description:
            - Performs forward propagation through the network to compute the output layer activations.
            - Returns the final layer activations as predictions.

        Example:
            predictions = nn.predict(x_test)
        """

        results = []
        for x in input:
            # print(x,"\n")
            z0 = self.activation_f(self.weights[0] @ x + self.bias[0])
            activations = [z0]
            for l in range(1, self.layer_n - 1):
                zl = self.weights[l] @ activations[l - 1] + self.bias[l]
                a = self.activation_f(zl)
                activations.append(a)

            results.append(activations[-1])

        return results

In [85]:
def quantum_tanh(z,a, ϵ=None):

   """

   """

   return np.tanh(z/a)

def rotate(zs: np.ndarray[:],
           states: np.ndarray[:],
           W: list[np.ndarray[:,:] ], 
           σ: callable = quantum_tanh ,
           a: (int | float ) = .5) -> np.ndarray[:,:]: 

   # determine angle of rotation 
   new_states = np.zeros((W.shape[0],len(states[0])))
   θ = π/2*(zs-σ(np.sum(W@zs),a))
   # actually rotate states 
   R = np.array([np.cos(θi/2)*np.eye(2)-j*np.sin(θi/2)*σy for θi in θ])
   for i in range(len(new_states)):
      new_states[i] = R[i]@states[i]

   return new_states


In [86]:
rotate(zs,states, W)

/var/folders/04/2cqfhcv133s3gbkf3b35tnkr0000gn/T/ipykernel_15937/3287989942.py:16: ComplexWarning: Casting complex values to real discards the imaginary part
  new_states[i] = R[i]@states[i]


IndexError: index 3 is out of bounds for axis 0 with size 3

In [137]:
rotate(zs, states, W)

AttributeError: 'list' object has no attribute 'shape'

In [143]:
n = 3 
x =  np.array([[1.0,0.0],[1.0,0.0]],dtype=complex)
y =  np.array([[0.0,1.0],[1.0,0.0]],dtype=complex)
w0 = np.random.rand(n,len(x))
w1 = np.random.rand(n,n)
w2 = np.random.rand(n,n)

x[0]
W = [w0 , w1, w2]

classical_bits = W[0] @ x
classical_bits = classical_bits / np.linalg.norm(classical_bits, axis=1, keepdims=True)
measurements, initial_states = measure(classical_bits,spin_half_values)
print(initial_states)

# intialize all to up state at first 
state_list = [initial_states] + [ np.zeros((W[l].shape[0],len(spin_half_values)),dtype=complex) for l in range(len(W))]
state_list = intialize_up(W,state_list)

# perform measure-rotate feedforward
for l in range(1,len(W)-1): 
    states = rotate(measurements,initial_states,W[l])
    print(f"state {l}:",states,end="\n")
    measurements, states = measure(states,spin_half_values)
    state_list[l] = states


print(state_list[-1])

# final linear classical step
classical_output = W[-1] @ state_list[-1]

print(classical_output)

[[1.+0.j 0.+0.j]
 [1.+0.j 0.+0.j]
 [1.+0.j 0.+0.j]]
state 1: [[ 0.92467318+0.j -0.38076175+0.j]
 [ 0.92467318+0.j -0.38076175+0.j]
 [ 0.92467318+0.j -0.38076175+0.j]]
[[1.+0.j 0.+0.j]
 [1.+0.j 0.+0.j]
 [1.+0.j 0.+0.j]]
[[1.45344469+0.j 0.        +0.j]
 [0.8313436 +0.j 0.        +0.j]
 [1.63428869+0.j 0.        +0.j]]


In [43]:
W[0] @ x

array([[0.87491866+0.j, 0.        +0.j],
       [1.12090617+0.j, 0.        +0.j],
       [1.05818881+0.j, 0.        +0.j]])

In [37]:
x.T

array([[1.+0.j, 0.+0.j],
       [0.+0.j, 1.+0.j]])

In [239]:

# introduce classical step : 
classical_bits = W[0] @ x
# measurements, initial_states = measure(classical_bits,eigenvalues)
classical_bits


array([[0.60699857+0.j, 0.74329641+0.j],
       [0.01747205+0.j, 0.9929374 +0.j],
       [0.26924902+0.j, 0.81881046+0.j]])

In [164]:
type(spin_half_values[0])

int

In [141]:
type(1.0+0j)

complex

In [166]:
eig_index = range(len(spin_half_values))
ps = initial_state[0]*initial_state[0]
print(ps)
index = np.random.choice(eig_index,p=ps) 

[1.+0.j 0.+0.j]


TypeError: Cannot cast array data from dtype('complex128') to dtype('float64') according to the rule 'safe'

In [ ]:
  # introduce classical step : 
classical_bits = W[0] @ x
measurements, initial_states = measure(classical_bits,spin_half_values)

ValueError: probabilities do not sum to 1

In [246]:
eigenvalues = spin_half_values
state_vectors = classical_bits

eig_index = range(len(eigenvalues))  # Indices of eigenvalues
    # Measurements: stores the results (e.g., energy, spin) for each state
measurements = np.zeros(len(state_vectors), dtype=float)
    # Projected states: stores the new wavefunctions after measurement
projected_states = np.zeros((len(state_vectors), len(eigenvalues)), dtype=complex)
for i in range(len(state_vectors)): 
        # Compute probability distribution from the state vector amplitudes
    ps = np.abs(state_vectors[i])**2  # Probabilities must be real and non-negative
        # Randomly choose an eigenvalue index according to probability distribution
    index = np.random.choice(eig_index, p=ps) 
        # Store the measured eigenvalue
    measurements[i] = eigenvalues[index]
        # Collapse the wavefunction: Set to the measured eigenstate
    projected_states[i] = np.zeros(len(eigenvalues), dtype=complex)  # Reset state
    projected_states[i, index] = 1.0 + 0.0j  # Collapse to the measured eigenstate





ValueError: probabilities do not sum to 1

In [137]:
def measure(state_vectors: np.ndarray, eigenvalues: np.ndarray):
    """
    Perform a quantum measurement on a set of state vectors.

    Parameters:
    - state_vectors (np.ndarray): A 2D NumPy array of shape (N, M), where N is the number of state vectors
      and M is the number of possible eigenstates. Each row represents a quantum state in an eigenbasis.
    - eigenvalues (np.ndarray): A 1D NumPy array of shape (M,) containing the eigenvalues corresponding 
      to the measurement observable.

    Returns:
    - measurements (np.ndarray): A 1D NumPy array of shape (N,) containing the measured eigenvalues 
      for each state vector.
    - projected_states (np.ndarray): A 2D NumPy array of shape (N, M) representing the post-measurement 
      states (collapsed wavefunctions) after measurement.
    """

    eig_index = np.arange(len(eigenvalues))  # Indices of eigenvalues
    # Measurements: stores the results (e.g., energy, spin) for each state
    measurements = np.zeros(len(state_vectors), dtype=float)
    # Projected states: stores the new wavefunctions after measurement
    projected_states = np.zeros((len(state_vectors), len(eigenvalues)), dtype=complex)
    for i in range(len(state_vectors)): 
        # Compute probability distribution from the state vector amplitudes
        ps = np.abs(state_vectors[i])**2  # Probabilities must be real and non-negative
        # Randomly choose an eigenvalue index according to probability distribution
        index = np.random.choice(eig_index, p=ps) 
        # Store the measured eigenvalue
        measurements[i] = eigenvalues[index]
        # Collapse the wavefunction: Set to the measured eigenstate
        projected_states[i] = np.zeros(len(eigenvalues), dtype=complex)  # Reset state
        projected_states[i, index] = 1.0 + 0.0j  # Collapse to the measured eigenstate

    return measurements, projected_states


def test_classical_step():
    
    """
    tests whether n qubits can be multipled by a weight matrix 

    e.g. we have 2 qubits, and we want three classical outputs , 
    these form "eigenvalues fed into rotation" 

    """


def test_measure():
    
    # DO NOT READ THIS PART AS THE CLASSICAL STEP
    N_tests = 500000
    # 3 qubits going into 2 gates via classical step 
    x = np.array([[1.0,0.0],[1.0,0.0],[0.0,1.0]],dtype=complex)
    W = np.array([[1.0,2.0,3.0],[4.0,5.0,6.0]],dtype=complex)

    qubits = W @ x
    assert np.allclose(W @ x, np.array([[3.0,3.0],[9.0,6.0]],dtype=complex))

    # normalize 
    qubits = (W @ x) / np.linalg.norm(qubits, axis=1, keepdims=True)

    measurements, states = measure(qubits,spin_half_values)
    assert len(measurements) == qubits.shape[0]
    assert states.shape == qubits.shape

    eigenvalues = spin_half_values
    state_vectors = qubits
    eig_index = range(len(eigenvalues))  # Indices of eigenvalues
    # Measurements: stores the results (e.g., energy, spin) for each state
    measurements = np.zeros(len(state_vectors), dtype=float)
    # Projected states: stores the new wavefunctions after measurement
    projected_states = np.zeros((len(state_vectors), len(eigenvalues)), dtype=complex)

    tests = np.zeros((N_tests,len(state_vectors), len(eigenvalues)),dtype=complex)
    for test_index in range(N_tests):
        for i in range(len(state_vectors)): 
            ps = np.abs(state_vectors[i])**2 
            # Randomly choose an eigenvalue index according to probability distribution
            index = np.random.choice(eig_index, p=ps) 
            # Store the measured eigenvalue
            measurements[i] = eigenvalues[index]
            # Collapse the wavefunction: Set to the measured eigenstate
            projected_states[i] = np.zeros(len(eigenvalues), dtype=complex)  # Reset state
            projected_states[i, index] = 1.0 + 0.0j  # Collapse to the measured eigenstate

        tests[test_index,:,:] = projected_states  

    # now test measure follows prob distribution 
    ps_first_state = np.abs(state_vectors[0])**2
    ps_second_state = np.abs(state_vectors[1])**2

    # only take spin up states
    ps = np.array([ps_first_state[0], ps_second_state[0]])
    print("Quantum Probabilities:", ps)
    # determine fraction of spin up for first state , should be roughly half
    tested_ps = np.array([np.sum(tests[:, 0, 0] == 1)/N_tests, np.sum(tests[:, 1, 0] == 1)/N_tests])
    print(f"Measured Probabilities: {tested_ps}")
    assert np.allclose(ps,tested_ps, atol=1e-3)

test_measure()

Quantum Probabilities: [0.5        0.69230769]
Measured Probabilities: [0.499782 0.692336]


In [247]:
# 3 qubits going into 2 gates via classical step 
x = np.array([1.0,0.0],dtype=complex)

w = np.array([[1.0,2.0],
              [3.0,4.0],
              [5.0,6.0]],dtype=complex)

w1 = np.array ([[1.0,2.0,3.0 ],
                [1.0,2.0,3.0 ],
                [1.0,2.0,3.0 ]],dtype=complex)

w2 = np.array([[1.0,2.0,3.0 ],
               [1.0,2.0,3.0 ],
               [1.0,2.0,3.0 ]],dtype=complex )

w3= np.array([[1.0,2.0,3.0 ],
               [1.0,2.0,3.0 ]],dtype=complex )

W = [w,w1,w2,w3]

classical_measurements = W[0] @ x
classical_measurements = (classical_measurements) / np.linalg.norm(classical_measurements, 
                                                                   keepdims=True)

print(f"classical_step results:",classical_measurements,end="\n")

state_list = [ np.zeros((W[l].shape[0],len(spin_half_values)),
                        dtype=complex) for l in range(len(W)-1)]



initial_state = intialize_up(W[1:-1],state_list)


print("initial_state:",len(initial_state))
# perform measure-rotate feedforward
for l in range(1,len(W)-1): 
    states = rotate(measurements,initial_state[l],W[l])
    measurements, states = measure(states,spin_half_values)
    print(f"{l}:",states)
    state_list[l] = states

print(state_list[-1])
    # final linear classical step
# classical_output = W[-1] @ state_list[-1]
# # new_states

# eigenvalues = spin_half_values
# state_vectors = classical_bits

# print(state_vectors)
# eig_index = range(len(eigenvalues))  # Indices of eigenvalues
#     # Measurements: stores the results (e.g., energy, spin) for each state
# measurements = np.zeros(len(state_vectors), dtype=float)
#     # Projected states: stores the new wavefunctions after measurement
# projected_states = np.zeros((len(state_vectors), len(eigenvalues)), dtype=complex)
# for i in range(len(state_vectors)): 
#         ps = np.abs(state_vectors[i])**2 
#         # Randomly choose an eigenvalue index according to probability distribution
#         index = np.random.choice(eig_index, p=ps) 
#         # Store the measured eigenvalue
#         measurements[i] = eigenvalues[index]
#         # Collapse the wavefunction: Set to the measured eigenstate
#         projected_states[i] = np.zeros(len(eigenvalues), dtype=complex)  # Reset state
#         projected_states[i, index] = 1.0 + 0.0j  # Collapse to the measured eigenstate
#    # determine angle of rotation 

# print(projected_states)
# print(projected_states,measurements)

# θ = π/2*(measurements-quantum_tanh(np.sum(W[1]@measurements),.1))
# print(θ)
# new_states = np.zeros((W[1].shape[0],len(projected_states[0])),dtype=complex)

# # actuallprint(θ)
# R = np.array([np.cos(θi/2)*np.eye(2)-j*np.sin(θi/2)*σy for θi in θ],dtype=complex)
# for i in range(len(new_states)):
#     new_states[i] = R[i]@states[i]

# new_states

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.

We use the term queron as an abbreviation for quantum neuron which stores a vector, a vector representing a *state* . Now for sake of efficiency we represent the set of all states at any layer by a matrix , a *state matrix*, and rows constitute *a* state.

Let us follow the feedforward process. There are two ways to represent the input data.  

The data can be encoded *classically*, ones and zeros. However, it can easily be encoded in terms of *qubits*. One is spin-up, zero is spin-down. 

If we take the second approach. For the first step , between input and the first measurement, we apply a classical linear step. A weight matrix acts on the *state matrix* in the classical case as:

$W_{ij}$ is the connection between the jth neuron in the (l-1)th layer and the ith neuron in the lth layer. 

However, furthermore we have $W_{ijk}$ which is the connection of the kth element in the (l-1)th layer of the jth qeuron to the i element of the weight matrix. 

So we have an equation:

(1.1) $W_{ij}^l * Q_{jk}^l$ = $Q_{k}^{l+1}$

However using this approach it becomes complicated to feed from a layer with m elemenbts to a layer with n elements. 

If we follow the first approach. We take classical input data and we multiply it by a weight matrix which gives us the number of activations we need to perform an initial rotation.  